In [1]:
#Figure 7,code line 639,P11

import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, LSTM, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import matplotlib.pyplot as plt

def load_data(folders, labels, base_path, frame_count=10):
    data = []
    targets = []
    for folder in folders:
        folder_path = os.path.join(base_path, folder)
        for video_folder in os.listdir(folder_path):
            video_path = os.path.join(folder_path, video_folder)
            frames = []
            for frame_file in sorted(os.listdir(video_path))[:frame_count]:
                frame_path = os.path.join(video_path, frame_file)
                frame = cv2.imread(frame_path)
                frame = cv2.resize(frame, (64, 64))
                frames.append(frame)
            if len(frames) < frame_count:
                frames.extend([np.zeros_like(frames[0])]*(frame_count - len(frames)))
            data.append(frames)
            targets.append(labels[folder])
    return np.array(data), np.array(targets)

#需要改文件路径
base_path = "E:/2024t2/5925/violence-detection-dataset"
folders = ["high-level violence_frames", "low-level violence_frames", "non-violence_frames"]
labels = {"high-level violence_frames": 0, "low-level violence_frames": 1, "non-violence_frames": 2}

data, targets = load_data(folders, labels, base_path)
data = data / 255.0

def create_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(inputs)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(x)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(x)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = TimeDistributed(Flatten())(x)
    x = LSTM(64)(x) 
    x = Dense(256, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

n_runs = 30
learning_rates = [0.01, 0.001, 0.0001]

results_lr = {f'learning_rate_{lr}': {'train_accuracies': [], 'test_accuracies': [], 'f1_scores': [], 'auc_scores': []} for lr in learning_rates}

for lr in learning_rates:
    for run in range(n_runs):
        X_train, X_val, y_train, y_val = train_test_split(data, targets, test_size=0.3, random_state=run)
        
        model = create_model((10, 64, 64, 3), 3)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=16, verbose=0)
        train_accuracy = history.history['accuracy'][-1]
        y_pred_prob = model.predict(X_val)
        y_pred = np.argmax(y_pred_prob, axis=1)
        test_accuracy = np.mean(y_pred == y_val)
        f1 = f1_score(y_val, y_pred, average='weighted')
        auc = roc_auc_score(y_val, y_pred_prob, multi_class='ovr')
        
        results_lr[f'learning_rate_{lr}']['train_accuracies'].append(train_accuracy)
        results_lr[f'learning_rate_{lr}']['test_accuracies'].append(test_accuracy)
        results_lr[f'learning_rate_{lr}']['f1_scores'].append(f1)
        results_lr[f'learning_rate_{lr}']['auc_scores'].append(auc)


for lr in learning_rates:
    mean_train_accuracy = np.mean(results_lr[f'learning_rate_{lr}']['train_accuracies'])
    mean_test_accuracy = np.mean(results_lr[f'learning_rate_{lr}']['test_accuracies'])
    mean_f1_score = np.mean(results_lr[f'learning_rate_{lr}']['f1_scores'])
    mean_auc_score = np.mean(results_lr[f'learning_rate_{lr}']['auc_scores'])

    print(f"Results for Learning Rate {lr}:")
    print(f"Average Train Accuracy: {mean_train_accuracy}")
    print(f"Average Test Accuracy: {mean_test_accuracy}")
    print(f"Average F1 Score: {mean_f1_score}")
    print(f"Average AUC Score: {mean_auc_score}\n")


KeyboardInterrupt: 

In [ ]:
#0926
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, LSTM, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
import matplotlib.pyplot as plt

def load_data(folders, labels, base_path, frame_count=10):
    data = []
    targets = []
    for folder in folders:
        folder_path = os.path.join(base_path, folder)
        for video_folder in os.listdir(folder_path):
            video_path = os.path.join(folder_path, video_folder)
            frames = []
            for frame_file in sorted(os.listdir(video_path))[:frame_count]:
                frame_path = os.path.join(video_path, frame_file)
                frame = cv2.imread(frame_path)
                frame = cv2.resize(frame, (64, 64))
                frames.append(frame)
            if len(frames) < frame_count:
                frames.extend([np.zeros_like(frames[0])]*(frame_count - len(frames)))
            data.append(frames)
            targets.append(labels[folder])
    return np.array(data), np.array(targets)

# 需要修改文件路径
base_path = "E:/2024t2/5925/violence-detection-dataset"
folders = ["high-level violence_frames", "low-level violence_frames", "non-violence_frames"]
labels = {"high-level violence_frames": 0, "low-level violence_frames": 1, "non-violence_frames": 2}

data, targets = load_data(folders, labels, base_path)
data = data / 255.0

def create_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(inputs)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(x)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = Conv3D(64, (3, 3, 3), activation='relu', padding='same')(x)  
    x = MaxPooling3D((2, 2, 2))(x)
    x = TimeDistributed(Flatten())(x)
    x = LSTM(64)(x) 
    x = Dense(256, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model

n_runs = 30
learning_rates = [0.01, 0.001, 0.0001]

results_lr = {f'learning_rate_{lr}': {'train_accuracies': [], 'test_accuracies': [], 'f1_scores': [], 'auc_scores': []} for lr in learning_rates}

for lr in learning_rates:
    for run in range(n_runs):
        X_train, X_val, y_train, y_val = train_test_split(data, targets, test_size=0.3, random_state=run)
        
        model = create_model((10, 64, 64, 3), 3)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=16, verbose=0)
        train_accuracy = history.history['accuracy'][-1]
        y_pred_prob = model.predict(X_val)
        y_pred = np.argmax(y_pred_prob, axis=1)
        test_accuracy = np.mean(y_pred == y_val)
        f1 = f1_score(y_val, y_pred, average='weighted')
        auc = roc_auc_score(y_val, y_pred_prob, multi_class='ovr')
        
        results_lr[f'learning_rate_{lr}']['train_accuracies'].append(train_accuracy)
        results_lr[f'learning_rate_{lr}']['test_accuracies'].append(test_accuracy)
        results_lr[f'learning_rate_{lr}']['f1_scores'].append(f1)
        results_lr[f'learning_rate_{lr}']['auc_scores'].append(auc)

for lr in learning_rates:
    mean_train_accuracy = np.mean(results_lr[f'learning_rate_{lr}']['train_accuracies'])
    mean_test_accuracy = np.mean(results_lr[f'learning_rate_{lr}']['test_accuracies'])
    mean_f1_score = np.mean(results_lr[f'learning_rate_{lr}']['f1_scores'])
    mean_auc_score = np.mean(results_lr[f'learning_rate_{lr}']['auc_scores'])
    
    var_train_accuracy = np.var(results_lr[f'learning_rate_{lr}']['train_accuracies'])
    var_test_accuracy = np.var(results_lr[f'learning_rate_{lr}']['test_accuracies'])
    var_f1_score = np.var(results_lr[f'learning_rate_{lr}']['f1_scores'])
    var_auc_score = np.var(results_lr[f'learning_rate_{lr}']['auc_scores'])

    print(f"Results for Learning Rate {lr}:")
    print(f"Average Train Accuracy: {mean_train_accuracy}, Variance: {var_train_accuracy}")
    print(f"Average Test Accuracy: {mean_test_accuracy}, Variance: {var_test_accuracy}")
    print(f"Average F1 Score: {mean_f1_score}, Variance: {var_f1_score}")
    print(f"Average AUC Score: {mean_auc_score}, Variance: {var_auc_score}\n")
